# Blog Search Agent — Mode Testing

Interactive notebook to test both modes of the blog search agent:

1. **Chat mode** — conversational, free-form natural language, multi-turn capable
2. **Workflow mode** — deterministic pipeline, structured JSON output with Pydantic validation

### Prerequisites
- AWS credentials configured (`AWS_PROFILE` or env vars)
- `.env` file with `GATEWAY_URL`, `COGNITO_*` vars (or `GATEWAY_TOKEN`)
- `uv sync` to install dependencies

In [ ]:
import sys
import os
import json
import asyncio
import logging
from pathlib import Path

# Ensure blog_search is importable
sys.path.insert(0, str(Path.cwd()))

# Suppress noisy loggers
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
for noisy in ["botocore", "urllib3", "httpcore", "httpx", "mcp.client",
              "strands.tools.mcp", "strands.models", "strands.agent.agent_executor",
              "strands.agent.event_loop", "strands.telemetry"]:
    logging.getLogger(noisy).setLevel(logging.WARNING)

In [ ]:
from blog_search_agent import (
    create_chat_agent,
    run_blog_search_workflow,
    extract_json_from_response,
)
from models import EVENT_TYPES, validate_response

print(f"Event types: {EVENT_TYPES}")

---
## 1. Chat Mode

Chat mode is conversational — it accepts free-form questions and responds with cited, grounded answers.
The agent has access to: `registry_lookup`, `rss_fetch`, `web_fetch`, `gateway__WebSearch`.

### 1.1 Basic event search

In [ ]:
# Create a chat agent (no memory — local testing without AgentCore session)
chat_agent = create_chat_agent(debug=True)

# Ask about recent events for a team
response = chat_agent("Find any recent injury news for Colgate Raiders basketball")
print("\n" + "="*70)
print("FINAL RESPONSE:")
print("="*70)
print(str(response))

### 1.2 Follow-up question (same agent instance = same conversation context)

In [ ]:
# Follow-up on the same agent — tests multi-turn within a session
followup_response = chat_agent("Can you give me more details on the most recent event you found?")
print("\n" + "="*70)
print("FOLLOW-UP RESPONSE:")
print("="*70)
print(str(followup_response))

### 1.3 Out-of-scope request

In [ ]:
# Out-of-scope: should respond politely and explain what it can help with
oos_agent = create_chat_agent(debug=False)
oos_response = oos_agent("What's the weather like in New York today?")
print(str(oos_response))

### 1.4 Greeting (conversational handling)

In [ ]:
# Greeting: should respond conversationally, not attempt tool calls
greet_agent = create_chat_agent(debug=False)
greet_response = greet_agent("Hi! What can you help me with?")
print(str(greet_response))

### 1.5 Specific event type request

In [ ]:
# Ask for a specific event type
specific_agent = create_chat_agent(debug=True)
specific_response = specific_agent(
    "Are there any schedule changes or venue changes for Duke Blue Devils in the past 2 weeks?"
)
print("\n" + "="*70)
print("RESPONSE:")
print("="*70)
print(str(specific_response))

---
## 2. Workflow Mode

Workflow mode is a deterministic pipeline for batch/scheduled runs:
1. Fetches team blog URLs from DynamoDB registry
2. Runs a focused agent (RSS + web search) with strict structured output
3. Parses & validates with Pydantic (retries on failure)
4. Filters results by date window

### 2.1 Standard workflow run (today only — default)

In [ ]:
result = await run_blog_search_workflow(
    team="Colgate Raiders",
    sport="NCAA Men's Basketball",
    lookback_days=0,  # today only (default for workflow mode)
    debug=True,
)

print("\n" + "="*70)
print("WORKFLOW RESULT:")
print("="*70)
print(json.dumps(result, indent=2, default=str))

### 2.2 Validate the output schema

In [ ]:
if result.get("status") == "success":
    # Re-validate the returned data against Pydantic models
    validation = validate_response({
        "results": result["results"],
        "retrieval_diagnostics": result["retrieval_diagnostics"],
    })
    if isinstance(validation, list):
        print("VALIDATION ERRORS:")
        for err in validation:
            print(f"  - {err}")
    else:
        print(f"Validation PASSED: {len(validation.results)} events")
        for ev in validation.results:
            print(f"  [{ev.event_type}] {ev.summary[:80]}")
else:
    print(f"Workflow returned error: {result.get('error')}")

### 2.3 Workflow with specific event types

In [ ]:
# Request only INJURY and ROSTER — INJURY is always force-included
result_filtered = await run_blog_search_workflow(
    team="Duke Blue Devils",
    sport="NCAA Men's Basketball",
    events=["ROSTER"],  # INJURY will be auto-prepended
    lookback_days=0,  # today only
    debug=True,
)

print("\n" + "="*70)
print(f"Status: {result_filtered.get('status')}")
print(f"Results before date filter: {result_filtered.get('results_before_filter')}")
print(f"Results after date filter: {result_filtered.get('results_after_filter')}")
print(f"Date range: {result_filtered.get('date_range')}")
print("="*70)

for ev in result_filtered.get("results", []):
    print(f"  [{ev['event_type']}] {ev['summary'][:80]}")
    print(f"    Source: {ev['source_url'][:60]}")
    print(f"    Date: {ev.get('blog_post_date', 'unknown')}")
    print()

### 2.4 Workflow with no results (today only, unlikely to have events)

In [ ]:
# lookback_days=0 means today only — tests the empty-results path
result_empty = await run_blog_search_workflow(
    team="Chicago State Cougars",
    sport="NCAA Men's Basketball",
    lookback_days=0,
    debug=False,
)

print(f"Status: {result_empty.get('status')}")
print(f"Results: {len(result_empty.get('results', []))}")
print(f"Diagnostics: {json.dumps(result_empty.get('retrieval_diagnostics', {}), indent=2)}")

### 2.5 Workflow with non-existent team (error path)

In [ ]:
result_err = await run_blog_search_workflow(
    team="Nonexistent University Falcons",
    sport="NCAA Men's Basketball",
    lookback_days=0,  # today only
    debug=False,
)

print(f"Status: {result_err.get('status')}")
print(f"Error: {result_err.get('error', 'N/A')}")

---
## 3. Pydantic Validation (Unit Tests)

Test the validation logic directly without calling the agent.

In [ ]:
from models import BlogSearchResponse, EventResult, RetrievalDiagnostics

# Valid response
valid_payload = {
    "results": [
        {
            "sport": "NCAA Men's Basketball",
            "team": "Colgate Raiders",
            "event_type": "INJURY",
            "player_name": "John Smith",
            "excerpt": "Smith will miss 4-6 weeks with a knee injury sustained in practice.",
            "summary": "John Smith out 4-6 weeks with knee injury",
            "source_url": "https://colgatefanblog.com/2026/07/smith-injury",
            "blog_post_date": "2026-07-10",
            "detected_at": "2026-07-14T12:00:00Z",
            "retrieval_method": "rss + web_fetch",
        }
    ],
    "retrieval_diagnostics": {
        "urls_total": 3,
        "urls_with_rss": 2,
        "urls_without_rss": 1,
        "web_searches_performed": 1,
        "rss_feeds_fetched": 2,
        "web_fetches_performed": 1,
        "events_found_via_rss": True,
        "events_found_via_web_search": False,
    },
}

result = validate_response(valid_payload)
assert not isinstance(result, list), f"Expected valid, got errors: {result}"
print(f"Valid payload: PASSED ({len(result.results)} events)")
print(f"  event_type: {result.results[0].event_type}")
print(f"  retrieval_method: {result.results[0].retrieval_method}")

In [ ]:
# Invalid response — bad event_type and missing required field
invalid_payload = {
    "results": [
        {
            "sport": "NCAA Men's Basketball",
            "team": "Colgate Raiders",
            "event_type": "TRADE",  # invalid
            "excerpt": "Some text",
            "summary": "Something happened",
            "source_url": "https://example.com",
            # missing: detected_at, retrieval_method
        }
    ],
    "retrieval_diagnostics": {
        "urls_total": 1,
        "urls_with_rss": 0,
        "urls_without_rss": 1,
        "web_searches_performed": 1,
        "rss_feeds_fetched": 0,
        "web_fetches_performed": 0,
    },
}

result = validate_response(invalid_payload)
assert isinstance(result, list), "Expected validation errors"
print(f"Invalid payload: correctly rejected with {len(result)} error(s):")
for err in result:
    print(f"  - {err}")

In [ ]:
# Empty results (valid — no events found is acceptable)
empty_payload = {
    "results": [],
    "retrieval_diagnostics": {
        "urls_total": 2,
        "urls_with_rss": 1,
        "urls_without_rss": 1,
        "web_searches_performed": 1,
        "rss_feeds_fetched": 1,
        "web_fetches_performed": 0,
    },
}

result = validate_response(empty_payload)
assert not isinstance(result, list), f"Expected valid, got: {result}"
print(f"Empty results payload: PASSED (0 events, valid schema)")

---
## 4. JSON Extraction (Unit Tests)

Test the balanced-brace JSON extraction from agent response text.

In [ ]:
# JSON embedded in markdown code block
text_with_codeblock = '''Here are the results:
```json
{"results": [], "retrieval_diagnostics": {"urls_total": 0, "urls_with_rss": 0, "urls_without_rss": 0, "web_searches_performed": 0, "rss_feeds_fetched": 0, "web_fetches_performed": 0}}
```
'''
parsed = extract_json_from_response(text_with_codeblock)
assert parsed is not None
assert parsed["results"] == []
print("Code block extraction: PASSED")

# Raw JSON (no wrapping)
raw_json = '{"results": [{"sport": "Basketball"}], "retrieval_diagnostics": {"urls_total": 1, "urls_with_rss": 0, "urls_without_rss": 1, "web_searches_performed": 0, "rss_feeds_fetched": 0, "web_fetches_performed": 0}}'
parsed = extract_json_from_response(raw_json)
assert parsed is not None
assert len(parsed["results"]) == 1
print("Raw JSON extraction: PASSED")

# JSON with preamble text
text_with_preamble = 'Based on my search, here is the result: {"results": [], "retrieval_diagnostics": {"urls_total": 2, "urls_with_rss": 1, "urls_without_rss": 1, "web_searches_performed": 1, "rss_feeds_fetched": 1, "web_fetches_performed": 0}}'
parsed = extract_json_from_response(text_with_preamble)
assert parsed is not None
assert parsed["retrieval_diagnostics"]["urls_total"] == 2
print("Preamble extraction: PASSED")

# Non-JSON response (conversational)
conversational = "I couldn't find any recent injury news for that team. Would you like me to check another team?"
parsed = extract_json_from_response(conversational)
assert parsed is None
print("Non-JSON response: correctly returned None")

---
## 5. Summary

| Test | Mode | What it verifies |
|------|------|------------------|
| 1.1 | Chat | Basic event search with tool calls (7-day default lookback) |
| 1.2 | Chat | Follow-up query (multi-turn context) |
| 1.3 | Chat | Out-of-scope handling |
| 1.4 | Chat | Greeting / non-task handling |
| 1.5 | Chat | Specific event type filtering |
| 2.1 | Workflow | Standard pipeline (today only) |
| 2.2 | Workflow | Pydantic output validation |
| 2.3 | Workflow | Specific event types + INJURY auto-include (today only) |
| 2.4 | Workflow | Empty results path (today only) |
| 2.5 | Workflow | Non-existent team error handling |
| 3.x | Validation | Pydantic model unit tests |
| 4.x | Extraction | JSON parsing from agent text |